# 01. PyRPMD 势能面（PES）构建、验证与接口案例

本教程演示 **PyRPMD** 中势能面（Potential Energy Surface, PES）模块的四种核心工作模式：

1. **内置解析势能与 C++ 原生绑定 (`pyrpmd.potentials.builtin`)**：使用 `HarmonicPotential`、`MorsePairPotential`、`DoubleWellPotential`、`PeriodicCosinePotential`，进行批量坐标能量/力评估，并验证 C ABI v1 原生绑定与 `Dynamics` C++ 融合传播循环。
2. **自定义 Python 反应势能面与有限差分力检验 (`PythonPotential` & `check_potential`)**：加载或编写三原子交换反应 $\mathrm{A + BC \leftrightarrow AB + C}$ 解析教学势（`ToyExchange`），通过 `check_potential` 对比解析力 $-\nabla V(\mathbf{q})$ 与数值有限差分梯度，并扫描一维/二维反应路径。
3. **C ABI 编译型势能面构建与检验 (`build_object` & `CABIPotential`)**：将 C 语言源码编译为共享库（`.so`），通过 C ABI 接口直接调用并验证能量与解析力。
4. **环聚合物收缩（Ring-Polymer Contraction, RPC）组合势能 (`SplitPotential` & `DifferencePotential`)**：将快变参考势（在全部 $P$ 个珠子上评估）与慢变高阶修正势（收缩至 $P' < P$ 个珠子上评估）组合，降低昂贵从头算或机器学习势的调用开销。

> **注意**：本 Notebook 中的势能模型均为用于方法验证和教学演示的解析模型，所有文件输出均在临时目录中完成，不会污染工作区。

In [ ]:
from pathlib import Path
import tempfile
import json
import numpy as np

import pyrpmd
from pyrpmd import Dynamics, State, System, JobSpec
from pyrpmd.potentials import (
    HarmonicPotential,
    MorsePairPotential,
    DoubleWellPotential,
    PeriodicCosinePotential,
    CABIPotential,
    PESResult,
    check_potential,
    build_object,
    make_potential,
)
from pyrpmd.potentials.contraction import SplitPotential, DifferencePotential

# 自动定位 PyRPMD 仓库根目录
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path(pyrpmd.__file__).resolve().parents[2]

print(f"PyRPMD version: {pyrpmd.__version__}")
print(f"Repository root: {ROOT}")

## 1. 内置解析势能与 C++ 原生传播绑定

PyRPMD 内部统一采用**原子单位（Atomic Units）**：
- 坐标 $\mathbf{q}$：`bohr`，张量形状为 `[batch, natoms, 3]`
- 能量 $V(\mathbf{q})$：`hartree`，形状为 `[batch]`
- 力 $\mathbf{F}(\mathbf{q}) = -\nabla_{\mathbf{q}} V(\mathbf{q})$：`hartree / bohr`，形状为 `[batch, natoms, 3]`

当安装了 C++ 扩展时，内置势能可通过 `native_binding()` 向 `Dynamics` 提供零 Python 回调开销的 C ABI v1 指针。

In [ ]:
# 1.1 构造谐振子势、Morse 双原子势与双势阱模型
harmonic = HarmonicPotential(k=0.05)
morse = MorsePairPotential(atoms=(0, 1), depth=0.10, alpha=1.2, equilibrium=1.4)
double_well = DoubleWellPotential(height=0.02, width=1.5, confinement=0.05)

# 构造一批 (B=4, N=2, 3) 的坐标构型 (单位: bohr)
rng = np.random.default_rng(2026)
q_batch = np.zeros((4, 2, 3), dtype=np.float64)
q_batch[:, 1, 0] = np.linspace(1.2, 1.8, 4)  # 扫描两原子间距

harm_res = harmonic.evaluate(q_batch)
morse_res = morse.evaluate(q_batch)
dw_res = double_well.evaluate(q_batch)

# 有限差分检验解析力精度
morse_check = check_potential(morse, q_batch, epsilon=1e-6)
dw_check = check_potential(double_well, q_batch, epsilon=1e-6)

# 1.2 验证内置势在环聚合物动力学中的原生 C++ 执行比例
dyn = Dynamics(
    harmonic,
    masses=[1836.15, 1836.15],
    beta=1000.0,
    nbeads=16,
    dt=0.2,
    loop_backend="auto",
    threads=1,
)
q_ring = rng.normal(scale=0.1, size=(16, 2, 3))
state = State(q_ring, np.zeros_like(q_ring))
dyn.propagate(state, 50)

print(json.dumps({
    "morse_energies_hartree": np.round(morse_res.energy, 6).tolist(),
    "morse_max_force_error_au": morse_check["max_absolute_force_error"],
    "double_well_max_force_error_au": dw_check["max_absolute_force_error"],
    "native_binding_kind": harmonic.native_binding(natoms=2).kind,
    "dynamics_loop_stats": dyn.loop_stats,
}, indent=2))
assert morse_check["max_absolute_force_error"] < 1e-9
assert dw_check["max_absolute_force_error"] < 1e-9

## 2. 自定义 Python 反应势能面与二维扫描 (`ToyExchange`)

对于气相三原子交换反应 $\mathrm{A + BC \leftrightarrow AB + C}$，`examples/rate/toy_exchange.py` 提供了一个带两个渐近 Morse 通道和共线鞍点的解析势能面：
$$V(r_1, r_2, \theta) = D\left(e_1^2 - 2e_1 + e_2^2 - 2e_2\right) + A e_1 e_2 + B e_1 e_2 (1 + \cos\theta), \quad e_i = \exp[-\alpha(r_i - r_0)]$$

下面我们通过 `make_potential` 加载该势能，检验过渡态构型的受力与有限差分误差，并在共线平面 $(r_{\mathrm{AB}}, r_{\mathrm{BC}})$ 上扫描势能面轮廓。

In [ ]:
# 加载 examples/rate/pilot.inp 中定义的 ToyExchange 势能面与过渡态几何
pilot_spec = JobSpec.from_file(ROOT / "examples/rate/pilot.inp")
pilot_cfg, pilot_sys = pilot_spec.resolve()
toy_pes = make_potential(pilot_cfg["potential"], pilot_sys)

# 在过渡态 (TS) 附近检验解析力与有限差分的一致性
ts_q = pilot_sys.geometry[None, :, :]
ts_eval = toy_pes.evaluate(ts_q)
ts_fd_check = check_potential(toy_pes, ts_q + rng.normal(scale=0.05, size=(8, 3, 3)))

# 扫描共线交换路径 (r_AB, r_BC) 网格
grid_r = np.linspace(1.2, 3.8, 35)
r_ab, r_bc = np.meshgrid(grid_r, grid_r, indexing="ij")
flat_ab, flat_bc = r_ab.ravel(), r_bc.ravel()

# 构造共线三原子坐标: A 在 (-r_AB, 0, 0), B 在 (0, 0, 0), C 在 (+r_BC, 0, 0)
q_scan = np.zeros((len(flat_ab), 3, 3), dtype=np.float64)
q_scan[:, 0, 0] = -flat_ab
q_scan[:, 2, 0] = flat_bc
v_scan = toy_pes.evaluate(q_scan).energy.reshape(r_ab.shape)

# 对称伸缩坐标 r_1 = r_2 鞍点位置: r_saddle = r_0 + ln((2D + A)/(2D)) / alpha
r_saddle = 1.2 + np.log((2 * 0.08 + 0.32) / (2 * 0.08)) / 1.2

print(json.dumps({
    "ts_energy_hartree": float(ts_eval.energy[0]),
    "ts_max_force_norm_au": float(np.linalg.norm(ts_eval.forces[0], axis=-1).max()),
    "analytic_collinear_saddle_bohr": float(r_saddle),
    "finite_difference_max_error_au": ts_fd_check["max_absolute_force_error"],
}, indent=2))
assert ts_fd_check["max_absolute_force_error"] < 1e-9

In [ ]:
# 可视化 ToyExchange 共线二维势能面等高线图 (若安装了 matplotlib)
import importlib.util

if importlib.util.find_spec("matplotlib") is not None:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 5))
    levels = np.linspace(v_scan.min(), np.percentile(v_scan, 85), 24)
    cs = ax.contourf(r_ab, r_bc, v_scan, levels=levels, cmap="viridis")
    fig.colorbar(cs, ax=ax, label="Potential Energy (hartree)")
    ax.plot(r_saddle, r_saddle, "r*", markersize=12, label=f"Saddle ({r_saddle:.2f} bohr)")
    ax.set_xlabel(r"$r_{\mathrm{AB}}$ (bohr)")
    ax.set_ylabel(r"$r_{\mathrm{BC}}$ (bohr)")
    ax.set_title("Collinear ToyExchange PES Contour")
    ax.legend()
    plt.tight_layout()
    plt.close(fig)
    print("Matplotlib figure generated successfully.")

## 3. C ABI 编译型势能面构建与检验 (`CABIPotential`)

对于用 C/C++ 或 Fortran (`ISO_C_BINDING`) 编写的高性能分子/周期势能，PyRPMD 提供了标准的 C ABI v1（分子势）和 C ABI v2（带晶胞与维里张量的周期势）接口。下面我们将仓库自带的 `native/examples/harmonic_pes.c` 结合 `examples/pes/pes.toml` 清单编译为动态库，并用 `CABIPotential` 加载验证。

In [ ]:
import subprocess
import shutil

cc = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
assert cc is not None, "C compiler required for C ABI example"

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    obj_path = tmp_path / "harmonic.o"
    manifest_path = tmp_path / "pes.toml"
    manifest_path.write_text((ROOT / "examples/pes/pes.toml").read_text())

    # 1. 编译 C 目标文件并通过 build_object 链接生成共享库
    subprocess.run(
        [cc, "-O2", "-std=c11", "-fPIC", "-I", str(ROOT / "native/include"),
         "-c", str(ROOT / "native/examples/harmonic_pes.c"), "-o", str(obj_path)],
        check=True,
    )
    so_path = build_object(obj_path, manifest_path, cache_directory=tmp_path)

    with CABIPotential(so_path, manifest=manifest_path) as cabi_pes:
        q_test = np.array([[[1.0, -2.0, 0.5]]], dtype=np.float64)
        out = cabi_pes.evaluate(q_test)
        fd = check_potential(cabi_pes, q_test)
        # V = 0.5 * (x^2 + y^2 + z^2), F = -q
        expected_v = 0.5 * np.sum(q_test**2, axis=(1, 2))
        np.testing.assert_allclose(out.energy, expected_v, rtol=1e-12)
        np.testing.assert_allclose(out.forces, -q_test, rtol=1e-12)

print(json.dumps({
    "cabi_energy_hartree": out.energy.tolist(),
    "cabi_forces_au": out.forces.tolist(),
    "cabi_fd_max_error": fd["max_absolute_force_error"],
}, indent=2))

## 4. 环聚合物收缩（RPC）组合势能 (`SplitPotential` & `DifferencePotential`)

在凝聚相或大分子 PIMD/RPMD 中，全珠数评估昂贵势能 $V_{\mathrm{full}}$ 的计算量正比于珠子数 $P$。通过环聚合物收缩（Ring-Polymer Contraction, RPC）：
- **参考快变势 $V_{\mathrm{ref}}$** 在全部 $P$ 个珠子上计算；
- **慢变修正势 $\Delta V = V_{\mathrm{full}} - V_{\mathrm{ref}}$** 通过傅里叶正规模式投影收缩到 $P' < P$ 个珠子上计算，再将力提升回 $P$ 个珠子。

In [ ]:
# 构造 P=16 珠子向 P'=4 珠子收缩的差值势 (Delta-learning / RPC)
p_full, p_contracted = 16, 4
ref_pes = HarmonicPotential(k=0.04)
high_level_pes = HarmonicPotential(k=0.055)
correction_pes = DifferencePotential(high_level_pes, ref_pes)

rpc_pes = SplitPotential(
    fast=ref_pes,
    slow=correction_pes,
    nbeads=p_full,
    contracted_beads=p_contracted,
)

# 当所有珠子重合于质心时，收缩势与全珠数精确势的能量与力完全一致
q_centroid = np.broadcast_to(np.array([[[0.3, -0.1, 0.2], [-0.2, 0.4, -0.1]]]), (p_full, 2, 3)).copy()
rpc_centroid = rpc_pes.evaluate(q_centroid)
exact_centroid = high_level_pes.evaluate(q_centroid)

np.testing.assert_allclose(rpc_centroid.energy.mean(), exact_centroid.energy.mean(), rtol=1e-12)
np.testing.assert_allclose(rpc_centroid.forces.mean(axis=0), exact_centroid.forces.mean(axis=0), rtol=1e-12)

print(json.dumps({
    "full_beads": p_full,
    "contracted_beads": p_contracted,
    "mean_rpc_energy_hartree": float(rpc_centroid.energy.mean()),
    "mean_exact_energy_hartree": float(exact_centroid.energy.mean()),
}, indent=2))